# BTL02 – Nhận dạng cảm xúc khuôn mặt FER2013 – CNN PyTorch (Nhóm 06)
Notebook chạy thực nghiệm trên **Google Colab GPU**. Mỗi thành viên mở notebook bằng tài khoản của mình, chạy **nhóm thực nghiệm mình phụ trách**.

**Trước khi chạy (1 lần):**
1. `Runtime → Change runtime type → T4 GPU`.
2. Thư mục chung trên Drive [`BTL02_EmotionRecognition/`](https://drive.google.com/drive/folders/1AKapzPmhCmMYonTZA2YqYAnzTFzx-JSo) (Kỳ chia sẻ quyền Editor cho cả nhóm; các bạn khác bấm *Organize → Add shortcut* vào **My Drive**). Trong đó có `data/fer2013.csv.zip`, `exps/`, file nhật ký Google Sheets.
3. Sửa ô **Cấu hình** bên dưới: tên người chạy và nhóm thực nghiệm.

Kết quả mỗi lần chạy tự lưu vào `BTL02_EmotionRecognition/exps/<nhóm>/<run_id>/` trên Drive (log, cấu hình, hình, trọng số) và cập nhật file nhật ký `BTL02_NhatKyThucNghiem.xlsx`.

| Thành viên | Nhóm thực nghiệm | Lệnh |
|---|---|---|
| Lê Châu Kỳ (TV1) | B – kỹ thuật huấn luyện | `--group B` |
| Trần Thanh Huyền (TV2) | A – kiến trúc CNN | `--group A` |
| Vũ Hoàng Phương (TV3) | C – kích hoạt, điều chuẩn | `--group C` |
| Đỗ Minh Thương (TV4) | D – học chuyển giao + chọn mô hình cuối | `--group D`, rồi `run_final.py` |

## 1. Cấu hình

In [ ]:
AUTHOR = "Lê Châu Kỳ"          # <-- tên người chạy (ghi vào nhật ký)
GROUP  = "B"                     # <-- A | B | C | D
REPO_URL = "https://github.com/lechauky/BTL02_EmotionRecognition.git"   # repo GitHub của nhóm
DRIVE_ROOT = "/content/drive/MyDrive/BTL02_EmotionRecognition"           # thư mục chung trên Drive
NOTE = ""                        # ghi chú cho lần chạy (vd 'chạy lại do mất kết nối')

## 2. Kết nối Google Drive, kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, os
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU – đổi runtime sang T4!')
os.makedirs(DRIVE_ROOT + '/exps', exist_ok=True); os.makedirs(DRIVE_ROOT + '/data', exist_ok=True)
print(os.listdir(DRIVE_ROOT + '/data'))

## 3. Lấy code mới nhất từ GitHub
(Nếu repo để private: tải zip code lên `DRIVE_ROOT/code`, giải nén và đổi `CODE` thành đường dẫn đó.)

In [ ]:
CODE = '/content/BTL02_EmotionRecognition'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO_URL} {CODE}
%cd {CODE}
!git log -1 --oneline
!pip -q install openpyxl
os.environ['FER_DATA_DIR'] = DRIVE_ROOT + '/data'
os.environ['FER_EXPS_DIR'] = DRIVE_ROOT + '/exps'
os.environ['FER_GITHUB_REPO'] = REPO_URL.removesuffix('.git')

## 4. Chuẩn bị dữ liệu (chỉ cần chạy 1 lần cho cả nhóm – tạo `data/fer2013.npz` và hình khảo sát dữ liệu)

In [ ]:
!python run_data.py

## 5. Chạy thử nhanh (2 epoch, 2.000 ảnh) để kiểm tra mọi thứ hoạt động
Lần chạy thử được đánh dấu `quick` và **không** đưa vào nhật ký chính thức.

In [ ]:
!python run_experiment.py --exp {GROUP}1 --quick --author "{AUTHOR}" 

## 6. Chạy cả nhóm thực nghiệm
Mỗi thực nghiệm (tối đa 60 epoch, dừng sớm sau 10 epoch không cải thiện) mất khoảng 5–40 phút trên T4. Nếu Colab ngắt giữa chừng, chạy lại riêng mã còn thiếu, ví dụ `--exp B3`.

In [ ]:
!python run_experiment.py --group {GROUP} --author "{AUTHOR}" --note "{NOTE}" 

## 7. Ghi nhật ký lên Google Sheets và xem kết quả
Đồng bộ bằng **tài khoản Google của người chạy**: sheet `NhatKy` được dựng lại từ `exps/` (không nhân đôi dòng, giờ Việt Nam), sheet `TongHop` được cập nhật. Thầy xem *File → Lịch sử phiên bản* để biết ai thêm dòng nào, lúc nào.

Nhật ký: https://docs.google.com/spreadsheets/d/17R3AmpbdE-dxEJrX0fMZhDw2iZWPCG3-ITbCVEo-V0E/edit

In [ ]:
# nạp lại code mới nhất (sau git pull) rồi đồng bộ
import importlib, src.config, make_log, src.tracking, src.gsheet
for m in (src.config, src.tracking, make_log, src.gsheet): importlib.reload(m)
src.gsheet.sync()

In [ ]:
import pandas as pd
!python make_log.py
df = pd.read_csv(os.environ['FER_EXPS_DIR'] + '/registry.csv')
df[['start','author','key','version','valid_acc','test_acc','best_epoch','train_time_s','run_id']].tail(20)

## 8. (TV4 – Đỗ Minh Thương) Chọn mô hình cuối sau khi cả 4 nhóm chạy xong

In [ ]:
!python run_final.py --author "{AUTHOR}" 

## 9. Đưa kết quả lên GitHub
Thư mục `exps/` trên Drive (trừ `checkpoints/*.pt`, đã có trong `.gitignore`) được chép về repo rồi commit. Trọng số `.pt` giữ trên Drive; link thư mục Drive ghi trong nhật ký.